# Phishing URL Detection - Initial Exploration

## Business Context

**Stakeholder**: Small security consulting firm (internal IT security team)

**Problem Statement**: Staff members are increasingly targeted by phishing attacks via suspicious URLs in emails and messages. The firm needs a quick-check tool where any employee can paste a URL and get an instant risk assessment.

**Business Goal**: 
- Reduce phishing click-through incidents by 70% within 6 months
- Provide real-time URL risk scoring with explainable features
- Enable rapid security awareness training by showing WHY a URL is risky (excessive subdomains, suspicious characters, unusual length, etc.)

**Success Criteria**: A lightweight classification model that can flag risky URLs with high precision, minimizing false alarms while catching genuine threats.

## CRISP-DM Methodology

We're following the **CRISP-DM** (Cross-Industry Standard Process for Data Mining) framework:

1. **Business Understanding** - Define the phishing detection problem and success metrics
2. **Data Understanding** (this notebook) - Load, explore, and identify key patterns in URL features
3. **Data Preparation** - Feature engineering, cleaning, train/test splits
4. **Modeling** - Build and tune classification models
5. **Evaluation** - Assess performance against business KPIs
6. **Deployment** - Create a simple prediction interface

**This session**: Phases 1-2 (Business Understanding + Data Understanding)

## 1. Environment Setup

In [ ]:
# Unpack the dataset archive
import zipfile
import os

# Extract datasets from archive
with zipfile.ZipFile('archive.zip', 'r') as zip_ref:
    zip_ref.extractall('data/')
    
print("Extracted files:")
for file in os.listdir('data/'):
    size_mb = os.path.getsize(f'data/{file}') / (1024 * 1024)
    print(f"  - {file} ({size_mb:.1f} MB)")

In [ ]:
# Load necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("Libraries loaded successfully")

In [ ]:
# Load the first dataset to get a feel for the data structure
df = pd.read_csv('data/dataset1.csv')

print(f"Dataset shape: {df.shape[0]:,} rows, {df.shape[1]} columns")
print(f"\nColumn names:")
print(df.columns.tolist())

## 2. Data Loading and Initial Inspection

In [ ]:
# Check all datasets - test loading with proper error handling
import glob
import warnings

dataset_files = sorted(glob.glob('data/dataset*.csv'))

print("Comparing all datasets:\n")
for file in dataset_files:
    issues = []
    
    try:
        # Try UTF-8 first
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            df_temp = pd.read_csv(file, encoding='utf-8', low_memory=False)
            if w:
                for warning in w:
                    if 'DtypeWarning' in str(warning.category):
                        issues.append("mixed dtypes")
    except UnicodeDecodeError:
        # Try latin-1 encoding
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                df_temp = pd.read_csv(file, encoding='latin-1', on_bad_lines='skip', low_memory=False)
                issues.append("encoding: latin-1")
        except Exception as e:
            print(f"{file}")
            print(f"  FAILED: {str(e)[:80]}")
            print()
            continue
    except Exception as e:
        # Handle parsing errors
        try:
            with warnings.catch_warnings(record=True) as w:
                warnings.simplefilter("always")
                df_temp = pd.read_csv(file, encoding='latin-1', on_bad_lines='skip', low_memory=False)
                issues.append("parsing errors")
        except Exception as e2:
            print(f"{file}")
            print(f"  FAILED: {str(e2)[:80]}")
            print()
            continue
    
    print(f"{file}")
    print(f"  Shape: {df_temp.shape[0]:,} rows × {df_temp.shape[1]} columns")
    print(f"  Columns: {df_temp.columns.tolist()[:5]}...")
    if issues:
        print(f"  Issues: {', '.join(issues)}")
    print()

## 3. Dataset Comparison and Analysis

## Key Finding: Heterogeneous Dataset Collection

**Discovery**: While the Kaggle page mentions "diverse approaches," loading the data reveals just how different these datasets are. This is the first time we're seeing the actual structure directly.

**What We Found**:
- 6 datasets with wildly different structures (32 to 112 columns)
- Sizes range from 10k to 235k rows
- Different feature engineering philosophies (some extract 100+ features, others focus on 14 core signals)
- dataset5 has data quality issues (parsing errors, mixed data types)

**Implication for Our Business Case**:

Our stakeholder needs **explainability** - when a URL is flagged, staff need to understand WHY. This means we need human-readable feature names, not just numerical indicators.

**Selection Criteria**:
1. Contains raw URL or domain column (for showing examples)
2. Feature names are interpretable (e.g., "NumDots" not "feature_47")
3. Sufficient training data (10k+ minimum)
4. Clean loading (no major data quality issues)

**Top Candidates**:
- **dataset3**: 11k rows, 89 features - has 'url', 'length_url', 'nb_dots' (interpretable)
- **dataset4**: 235k rows, 56 features - has 'URL', 'Domain', 'URLLength' (LARGEST, interpretable)
- **dataset6**: 10k rows, 50 features - has 'NumDots', 'SubdomainLevel' (interpretable but smallest)

**Decision**: Examine dataset3 and dataset4 in detail. dataset4's size (235k) is compelling for model performance, but we need to verify feature interpretability first.

## 4. Dataset4 Detailed Examination

In [ ]:
# Examine dataset4 in detail (largest candidate at 235k rows)
df4 = pd.read_csv('data/dataset4.csv')

print(f"=== Dataset4 Overview ===")
print(f"Shape: {df4.shape[0]:,} rows × {df4.shape[1]} columns\n")

# Create a DataFrame to display column names in a clean format
col_info = pd.DataFrame({
    'Column': df4.columns,
    'Type': df4.dtypes.astype(str).values
})

# Display first 15 and last 5 features
print("Features (showing first 15 and last 5):")
display(pd.concat([col_info.head(15), 
                  pd.DataFrame({'Column': ['...'], 'Type': ['...']}),
                  col_info.tail(5)])
       .style.set_caption("Dataset4 Feature Overview"))

In [ ]:
# Sample rows with key features to understand what the data captures
# Select a diverse subset of features that tell different parts of the story

key_features = [
    'URL',                    # What we're analyzing
    'Domain',                 # Domain extracted
    'URLLength',              # Structure: length
    'NoOfSubDomain',          # Structure: subdomain count
    'IsHTTPS',                # Security: HTTPS flag
    'IsDomainIP',             # Security: IP address instead of domain
    'Bank',                   # Content: banking keyword
    'Pay',                    # Content: payment keyword  
    'Crypto',                 # Content: crypto keyword
    'HasPasswordField',       # Behavior: has password input
    'HasObfuscation',         # Behavior: obfuscated characters
    'label'                   # Ground truth (0=legit, 1=phishing)
]

# Show 10 samples to get diverse examples
sample_df = df4[key_features].head(10)

# Set pandas display options for better readability
pd.set_option('display.max_colwidth', 60)
pd.set_option('display.width', None)

print("=== Sample URLs with Key Features ===\n")
print(sample_df.to_string(index=False))

# Reset display options
pd.reset_option('display.max_colwidth')
pd.reset_option('display.width')

In [ ]:
# Verify what the label values actually mean
# Check class distribution first
print("=== Label Distribution ===")
print(df4['label'].value_counts().sort_index())
print(f"\nClass balance:")
print(df4['label'].value_counts(normalize=True).sort_index())

print("\n=== Examples of label=0 ===")
label_0_sample = df4[df4['label'] == 0][['URL', 'Domain', 'IsHTTPS', 'HasObfuscation', 'Bank', 'label']].head(5)
print(label_0_sample.to_string(index=False))

print("\n=== Examples of label=1 ===")
label_1_sample = df4[df4['label'] == 1][['URL', 'Domain', 'IsHTTPS', 'HasObfuscation', 'Bank', 'label']].head(5)
print(label_1_sample.to_string(index=False))

## Correction: Label Encoding

**Previous Assumption (INCORRECT)**: In cell-8, we assumed `label=0` meant legitimate and `label=1` meant phishing.

**Evidence from verification**:
- **label=0** examples: `f0519141.xsph.ru`, `shprakserf.gq`, `kuradox92.lima-city.de` - suspicious domains with random strings, uncommon TLDs
- **label=1** examples: `uni-mainz.de`, `voicefmradio.co.uk`, `rewildingargentina.org` - recognizable legitimate organizations

**Actual encoding**:
- **0 = phishing** (42.8% of dataset, 100,945 samples)
- **1 = legitimate** (57.2% of dataset, 134,850 samples)

**Lesson**: Always verify assumptions about data encoding before proceeding with analysis. The comment in cell-8 remains as a reminder of this error.

## 5. Data Quality Assessment

In [ ]:
# Data quality check for dataset4
print("=== Dataset4 Data Quality Check ===\n")

# Check for missing values
print("Missing values per column:")
missing = df4.isnull().sum()
missing_cols = missing[missing > 0]
if len(missing_cols) > 0:
    print(missing_cols)
else:
    print("No missing values in any column")

print(f"\nTotal missing values: {df4.isnull().sum().sum()}")
print(f"Percentage of data missing: {(df4.isnull().sum().sum() / (df4.shape[0] * df4.shape[1]) * 100):.2f}%")

# Check for duplicates
print(f"\n=== Duplicate Check ===")
print(f"Duplicate rows: {df4.duplicated().sum()}")
print(f"Duplicate URLs: {df4['URL'].duplicated().sum()}")

# Basic data types check
print(f"\n=== Data Types ===")
print(f"Numeric columns: {len(df4.select_dtypes(include=['int64', 'float64']).columns)}")
print(f"Text columns: {len(df4.select_dtypes(include=['object']).columns)}")

## 6. Final Dataset Selection

## Dataset Selection Decision

Based on initial inspection, we focused on **dataset4** due to its size (235k samples - 2.5x larger than the next biggest). After examination, we confirm it meets our requirements:

**Why dataset4:**
1. **Size advantage**: 235,795 samples provide robust training data  
2. **Data completeness**: Zero missing values across all 56 columns
3. **Interpretable features**: Clear names that can be explained to users
4. **Contains raw URL and domain**: Essential for showing which URL was flagged
5. **Verified encoding**: Confirmed label meanings (0=phishing, 1=legitimate)

**Why NOT combine datasets:**
- Each dataset has completely different feature extraction approaches (32 to 112 columns)
- No standardized feature definitions across datasets
- Dataset4 alone is sufficient for our needs

**Note**: While dataset3 and dataset6 were identified as candidates, we proceeded with dataset4 after confirming it had all necessary characteristics. Further comparison was deemed unnecessary given dataset4's clear advantages.

# Dataset4 Feature Deep Dive

Now that we've selected dataset4, we need to understand what each feature actually measures. This understanding is critical for:

1. **Rule-based system** - Identifying clear red flags for instant detection
2. **ML model** - Understanding feature importance and model decisions  
3. **Explainability** - Explaining to users WHY a URL was flagged



## Structural Features

### Feature Investigation: Obfuscation Metrics

Three related features measure obfuscation in URLs:
- `HasObfuscation` (binary flag)
- `NoOfObfuscatedChar` (count)
- `ObfuscationRatio` (proportion)


In [ ]:
# Explore obfuscation features with better visualization
import matplotlib.pyplot as plt

# Create a cleaner crosstab display
print("=== Obfuscation Distribution by Label ===\n")
obf_crosstab = pd.crosstab(df4['HasObfuscation'], df4['label'], 
                           margins=True, margins_name="Total")
obf_crosstab.columns = ['Phishing (0)', 'Legitimate (1)', 'Total']
obf_crosstab.index = ['No Obfuscation', 'Has Obfuscation', 'Total']

# Display as a styled dataframe
display(obf_crosstab.style.set_caption("URL Obfuscation by Label")
        .format('{:,}')
        .set_table_styles([{'selector': 'caption', 
                           'props': [('font-size', '16px'), ('font-weight', 'bold')]}]))

# Show percentages
print("\n=== Percentage Distribution ===")
obf_pct = pd.crosstab(df4['HasObfuscation'], df4['label'], normalize='columns') * 100
obf_pct.columns = ['Phishing (0)', 'Legitimate (1)']
obf_pct.index = ['No Obfuscation %', 'Has Obfuscation %']
display(obf_pct.style.format('{:.2f}%')
        .background_gradient(cmap='YlOrRd', axis=None)
        .set_caption("Percentage of URLs with Obfuscation by Label"))

# Statistics summary in a cleaner format
print("\n=== Obfuscation Character Statistics ===")
stats = df4.groupby('label')['NoOfObfuscatedChar'].describe()[['mean', 'std', 'max']]
stats.index = ['Phishing (0)', 'Legitimate (1)']
display(stats.style.format('{:.4f}')
        .set_caption("Number of Obfuscated Characters Statistics"))

In [ ]:
# Show sample URLs with and without obfuscation
print("=== Sample URLs WITH Obfuscation (Phishing only) ===")
obf_samples = df4[df4['HasObfuscation'] == 1][['URL', 'NoOfObfuscatedChar', 'ObfuscationRatio']].head(5)
obf_samples['URL'] = obf_samples['URL'].str[:80] + '...'  # Truncate long URLs for display
display(obf_samples.style.set_caption("URLs with Obfuscation (all are phishing)")
        .format({'ObfuscationRatio': '{:.2%}'}))

print("\n=== Sample Legitimate URLs (Never have obfuscation) ===")
legit_samples = df4[df4['label'] == 1][['URL', 'Domain', 'IsHTTPS']].head(5)
display(legit_samples.style.set_caption("Sample Legitimate URLs"))

#### Obfuscation Findings

**Key Discovery**: Obfuscation is a **perfect indicator** of phishing in this dataset:
- **100% of legitimate URLs** have NO obfuscation
- Only **0.48% of phishing URLs** use obfuscation (485 out of 100,945)
- When present, obfuscation involves URL percent-encoding (e.g., `%20` for space, `%23` for #)

**Examples of obfuscation patterns found**:
- `banco%20davivienda` - encoding spaces in bank names
- `%26%29%24%21` - encoding special characters
- `%5bemail%5d` - encoding brackets around email placeholders

**Implication for rule-based system**: 
- `if HasObfuscation == 1: classify as PHISHING` (100% precision on this dataset)
- However, this only catches 0.48% of phishing URLs, so we need additional features

### Feature Investigation: IsDomainIP

This binary feature indicates whether the URL uses an IP address instead of a domain name.
Example: `http://192.168.1.1/login` vs `http://google.com/login`

In [ ]:
# Investigate IsDomainIP feature
print("=== IsDomainIP Distribution by Label ===\n")

# Create crosstab
ip_crosstab = pd.crosstab(df4['IsDomainIP'], df4['label'], 
                          margins=True, margins_name="Total")
ip_crosstab.columns = ['Phishing (0)', 'Legitimate (1)', 'Total']
ip_crosstab.index = ['Domain Name', 'IP Address', 'Total']

# Display counts
display(ip_crosstab.style.set_caption("URLs using IP vs Domain Name")
        .format('{:,}'))

# Show percentages
print("\n=== Percentage Distribution ===")
ip_pct = pd.crosstab(df4['IsDomainIP'], df4['label'], normalize='columns') * 100
ip_pct.columns = ['Phishing (0)', 'Legitimate (1)']
ip_pct.index = ['Domain Name %', 'IP Address %']
display(ip_pct.style.format('{:.2f}%')
        .background_gradient(cmap='RdYlGn_r', axis=None)
        .set_caption("Percentage of URLs using IP addresses"))

# Sample URLs with IP addresses
print("\n=== Sample URLs using IP addresses ===")
ip_samples = df4[df4['IsDomainIP'] == 1][['URL', 'Domain', 'label']].head(10)
ip_samples['URL'] = ip_samples['URL'].str[:60] + '...'
ip_samples['label'] = ip_samples['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(ip_samples.style.set_caption("Examples of URLs using IP addresses"))

#### IsDomainIP Findings

**Initial assumption**: Based on the feature name, I assume this indicates URLs using IP addresses instead of domain names.

**Verification from data**: Looking at sample URLs confirms this assumption - URLs with IsDomainIP=1 contain IP addresses in the Domain field.

**Key Discovery**: IP-based URLs are **extremely rare** and **only appear in phishing**:
- **100% of legitimate URLs** use domain names (0% use IP addresses)  
- Only **0.06% of phishing URLs** use IP addresses (58 out of 100,945)
- All 58 URLs with IP addresses are labeled as phishing

**Why this matters for phishing detection**:
- Legitimate websites use memorable domain names for branding and trust
- Phishers sometimes use IP addresses to avoid domain registration or to evade domain-based blocklists

**Implication for rule-based system**:
- `if IsDomainIP == 1: classify as PHISHING` (100% precision on this dataset)
- Like obfuscation, this is a perfect indicator but catches very few phishing URLs (0.06%)

### Feature Investigation: URLLength

I assume longer URLs might be associated with phishing (hiding the real domain with long paths/parameters).

In [ ]:
# Investigate URLLength distribution
import matplotlib.pyplot as plt

# Basic statistics
print("=== URLLength Statistics by Label ===\n")
url_stats = df4.groupby('label')['URLLength'].describe()
url_stats.index = ['Phishing (0)', 'Legitimate (1)']
display(url_stats.style.format('{:.1f}')
        .set_caption("URL Length Statistics"))

# Distribution plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist([df4[df4['label']==0]['URLLength'], 
              df4[df4['label']==1]['URLLength']], 
             bins=50, alpha=0.7, label=['Phishing', 'Legitimate'], color=['red', 'green'])
axes[0].set_xlabel('URL Length')
axes[0].set_ylabel('Count')
axes[0].set_title('URL Length Distribution by Label')
axes[0].legend()
axes[0].set_xlim(0, 200)  # Focus on main range

# Box plot - using tick_labels instead of labels for compatibility
box_data = [df4[df4['label']==0]['URLLength'], df4[df4['label']==1]['URLLength']]
bp = axes[1].boxplot(box_data, tick_labels=['Phishing', 'Legitimate'], patch_artist=True)
bp['boxes'][0].set_facecolor('salmon')
bp['boxes'][1].set_facecolor('lightgreen')
axes[1].set_ylabel('URL Length')
axes[1].set_title('URL Length Comparison')
axes[1].set_ylim(0, 200)  # Focus on main range

plt.tight_layout()
plt.show()

# Check extremes
print("\n=== Extreme URL Lengths ===")
print(f"Shortest phishing URL: {df4[df4['label']==0]['URLLength'].min()} chars")
print(f"Longest phishing URL: {df4[df4['label']==0]['URLLength'].max()} chars")
print(f"Shortest legitimate URL: {df4[df4['label']==1]['URLLength'].min()} chars")
print(f"Longest legitimate URL: {df4[df4['label']==1]['URLLength'].max()} chars")

In [ ]:
# Look at sample URLs by length categories
print("=== Sample URLs by Length Category ===\n")

# Very short URLs (< 20 chars)
print("VERY SHORT URLs (<20 chars):")
very_short = df4[df4['URLLength'] < 20][['URL', 'URLLength', 'label']].head(3)
very_short['label'] = very_short['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(very_short)

# Moderate length URLs (30-50 chars) 
print("\nMODERATE LENGTH URLs (30-50 chars):")
moderate = df4[(df4['URLLength'] >= 30) & (df4['URLLength'] <= 50)][['URL', 'URLLength', 'label']].head(3)
moderate['label'] = moderate['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(moderate)

# Very long URLs (>150 chars)
print("\nVERY LONG URLs (>150 chars):")
very_long = df4[df4['URLLength'] > 150][['URL', 'URLLength', 'label']].head(3)
very_long['URL'] = very_long['URL'].str[:80] + '...'  # Truncate for display
very_long['label'] = very_long['label'].map({0: 'Phishing', 1: 'Legitimate'})
display(very_long)

In [ ]:
# Verify exact URLLength statistics
phishing_urls = df4[df4['label'] == 0]['URLLength']
legit_urls = df4[df4['label'] == 1]['URLLength']

print("=== Exact URLLength Statistics ===")
print(f"\nPhishing URLs (label=0, n={len(phishing_urls)}):")
print(f"  Mean: {phishing_urls.mean():.1f} chars")
print(f"  Median: {phishing_urls.median():.1f} chars")
print(f"  Min: {phishing_urls.min()} chars")
print(f"  Max: {phishing_urls.max()} chars")

print(f"\nLegitimate URLs (label=1, n={len(legit_urls)}):")
print(f"  Mean: {legit_urls.mean():.1f} chars")  
print(f"  Median: {legit_urls.median():.1f} chars")
print(f"  Min: {legit_urls.min()} chars")
print(f"  Max: {legit_urls.max()} chars")

print(f"\nComparison:")
print(f"  Legitimate URLs are {legit_urls.mean() - phishing_urls.mean():.1f} chars longer on average")
print(f"  Legitimate median is {legit_urls.median() - phishing_urls.median():.1f} chars longer")

#### URLLength Findings

**Initial assumption**: I assumed longer URLs would be associated with phishing (to hide domains with complex paths).

**Actual findings from data** (verified in cell above):
- **Phishing URLs**: Mean 45.7 chars, Median 34.0 chars
- **Legitimate URLs**: Mean 26.2 chars, Median 26.0 chars  
- **My assumption was partially correct**: Phishing URLs ARE longer on average!

**Distribution insights**:
- Phishing URLs are 19.5 chars longer on average
- Phishing median is 8 chars longer than legitimate
- Phishing has extreme outliers (max: 6097 chars vs legitimate max: 57 chars)
- Legitimate URLs are very consistent (15-57 chars range)

**Why phishing URLs tend to be longer**:
- Complex redirects and tracking parameters
- Attempts to hide the real destination with long paths
- Subdomain spoofing (e.g., `secure.bank.com.evil.com/...`)
- URL shorteners that expand to long URLs

**Why legitimate URLs stay short**:
- Good UX practices favor short, memorable URLs
- SEO best practices recommend concise URLs
- Professional sites use proper domain names without complex tricks

**Implication for rule-based system**:
- URLs over 57 chars are ALWAYS phishing in this dataset (100% precision)
- Could use threshold: `if URLLength > 57: classify as PHISHING`
- This would catch many phishing URLs with perfect precision

### Feature Investigation: IsHTTPS

I assume legitimate sites are more likely to use HTTPS for security and trust signals.

In [ ]:
# Investigate IsHTTPS feature
print("=== HTTPS Usage by Label ===\n")

# Create crosstab
https_crosstab = pd.crosstab(df4['IsHTTPS'], df4['label'], 
                             margins=True, margins_name="Total")
https_crosstab.columns = ['Phishing (0)', 'Legitimate (1)', 'Total']
https_crosstab.index = ['HTTP', 'HTTPS', 'Total']

# Display counts
display(https_crosstab.style.set_caption("HTTP vs HTTPS Usage")
        .format('{:,}')
        .set_table_styles([{'selector': 'caption',
                           'props': [('font-size', '14px'), ('font-weight', 'bold')]}]))

# Show percentages
print("\n=== Percentage Distribution ===")
https_pct = pd.crosstab(df4['IsHTTPS'], df4['label'], normalize='columns') * 100
https_pct.columns = ['Phishing (%)', 'Legitimate (%)']
https_pct.index = ['HTTP', 'HTTPS']
display(https_pct.style.format('{:.1f}%')
        .background_gradient(cmap='RdYlGn', axis=1)
        .set_caption("Percentage of URLs using HTTPS"))

# Visualization
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 6))

# Stacked bar chart
labels = ['Phishing', 'Legitimate']
http_vals = [https_pct.iloc[0, 0], https_pct.iloc[0, 1]]
https_vals = [https_pct.iloc[1, 0], https_pct.iloc[1, 1]]

x = range(len(labels))
width = 0.5

p1 = ax.bar(x, http_vals, width, label='HTTP', color='#ff9999')
p2 = ax.bar(x, https_vals, width, bottom=http_vals, label='HTTPS', color='#90ee90')

ax.set_ylabel('Percentage (%)')
ax.set_title('HTTP vs HTTPS Usage by Label')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.legend()

# Add percentage labels on bars
for i, (http_val, https_val) in enumerate(zip(http_vals, https_vals)):
    ax.text(i, http_val/2, f'{http_val:.1f}%', ha='center', va='center')
    ax.text(i, http_val + https_val/2, f'{https_val:.1f}%', ha='center', va='center')

plt.tight_layout()
plt.show()

# Sample URLs
print("\n=== Sample HTTP Phishing URLs ===")
http_phishing = df4[(df4['IsHTTPS'] == 0) & (df4['label'] == 0)][['URL', 'Domain']].head(3)
display(http_phishing)

print("\n=== Sample HTTPS Phishing URLs ===")
https_phishing = df4[(df4['IsHTTPS'] == 1) & (df4['label'] == 0)][['URL', 'Domain']].head(3)
display(https_phishing)

In [ ]:
# Calculate exact percentages for verification
phishing_df = df4[df4['label'] == 0]
legit_df = df4[df4['label'] == 1]

print("=== Exact HTTPS Statistics ===")
print(f"\nPhishing URLs (label=0, n={len(phishing_df)}):")
print(f"  HTTP (IsHTTPS=0): {len(phishing_df[phishing_df['IsHTTPS']==0]):,} ({len(phishing_df[phishing_df['IsHTTPS']==0])/len(phishing_df)*100:.2f}%)")
print(f"  HTTPS (IsHTTPS=1): {len(phishing_df[phishing_df['IsHTTPS']==1]):,} ({len(phishing_df[phishing_df['IsHTTPS']==1])/len(phishing_df)*100:.2f}%)")

print(f"\nLegitimate URLs (label=1, n={len(legit_df)}):")
print(f"  HTTP (IsHTTPS=0): {len(legit_df[legit_df['IsHTTPS']==0]):,} ({len(legit_df[legit_df['IsHTTPS']==0])/len(legit_df)*100:.2f}%)")
print(f"  HTTPS (IsHTTPS=1): {len(legit_df[legit_df['IsHTTPS']==1]):,} ({len(legit_df[legit_df['IsHTTPS']==1])/len(legit_df)*100:.2f}%)")

#### IsHTTPS Findings

**Initial assumption**: I assumed legitimate sites would use HTTPS more for security and trust.

**Actual findings from data** (verified from cell above):
- **Phishing**: 50.78% HTTP, 49.22% HTTPS (almost evenly split)
- **Legitimate**: 0% HTTP, 100% HTTPS (ALL legitimate URLs use HTTPS)
- **Critical discovery**: ALL legitimate URLs in this dataset use HTTPS!

**Key insights**:
- **HTTPS is mandatory for legitimate sites** in this dataset (100% usage)
- **HTTP is a strong phishing indicator**: If HTTP, then 100% chance it's phishing in this dataset
- Almost half of phishing sites (49%) have adopted HTTPS to appear trustworthy

**Why this pattern exists**:
- Modern browsers mark HTTP sites as "Not Secure"
- Legitimate businesses migrated to HTTPS for SEO and trust
- Free SSL certificates (Let's Encrypt) made HTTPS accessible to phishers too

**Implication for rule-based system**:
- `if IsHTTPS == 0: classify as PHISHING` (100% precision in this dataset!)
- This catches 50.78% of phishing URLs (much better than obfuscation or IP features)
- Combined with other perfect indicators, we're building a strong rule set

### Feature Investigation: NoOfSubDomain

I assume phishing URLs might use more subdomains to create deceptive URLs like `paypal.com.secure.phishing-site.com`.

In [ ]:
# Investigate NoOfSubDomain feature
print("=== NoOfSubDomain Distribution ===\n")

# Get value counts for each label
phishing_subdomains = df4[df4['label'] == 0]['NoOfSubDomain'].value_counts().sort_index()
legit_subdomains = df4[df4['label'] == 1]['NoOfSubDomain'].value_counts().sort_index()

# Create comparison table
subdomain_comparison = pd.DataFrame({
    'Phishing Count': phishing_subdomains,
    'Legitimate Count': legit_subdomains
}).fillna(0).astype(int)

subdomain_comparison['Phishing %'] = (subdomain_comparison['Phishing Count'] / subdomain_comparison['Phishing Count'].sum() * 100).round(2)
subdomain_comparison['Legitimate %'] = (subdomain_comparison['Legitimate Count'] / subdomain_comparison['Legitimate Count'].sum() * 100).round(2)

display(subdomain_comparison.style
        .format({'Phishing Count': '{:,}', 'Legitimate Count': '{:,}',
                'Phishing %': '{:.2f}%', 'Legitimate %': '{:.2f}%'})
        .set_caption("Subdomain Count Distribution"))

# Visualization
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart comparison
x_vals = subdomain_comparison.index[:8]  # Focus on 0-7 subdomains
width = 0.35
x_pos = range(len(x_vals))

axes[0].bar([p - width/2 for p in x_pos], 
           subdomain_comparison.loc[x_vals, 'Phishing %'],
           width, label='Phishing', color='red', alpha=0.7)
axes[0].bar([p + width/2 for p in x_pos],
           subdomain_comparison.loc[x_vals, 'Legitimate %'],
           width, label='Legitimate', color='green', alpha=0.7)

axes[0].set_xlabel('Number of Subdomains')
axes[0].set_ylabel('Percentage (%)')
axes[0].set_title('Subdomain Distribution by Label')
axes[0].set_xticks(x_pos)
axes[0].set_xticklabels(x_vals)
axes[0].legend()

# Box plot
box_data = [df4[df4['label']==0]['NoOfSubDomain'], 
            df4[df4['label']==1]['NoOfSubDomain']]
bp = axes[1].boxplot(box_data, tick_labels=['Phishing', 'Legitimate'], patch_artist=True)
bp['boxes'][0].set_facecolor('salmon')
bp['boxes'][1].set_facecolor('lightgreen')
axes[1].set_ylabel('Number of Subdomains')
axes[1].set_title('Subdomain Count Comparison')

plt.tight_layout()
plt.show()

# Statistics
print("\n=== Subdomain Statistics ===")
print(f"Phishing - Mean: {df4[df4['label']==0]['NoOfSubDomain'].mean():.2f}, "
      f"Median: {df4[df4['label']==0]['NoOfSubDomain'].median():.0f}, "
      f"Max: {df4[df4['label']==0]['NoOfSubDomain'].max()}")
print(f"Legitimate - Mean: {df4[df4['label']==1]['NoOfSubDomain'].mean():.2f}, "
      f"Median: {df4[df4['label']==1]['NoOfSubDomain'].median():.0f}, "
      f"Max: {df4[df4['label']==1]['NoOfSubDomain'].max()}")

In [ ]:
# Show examples of URLs with different subdomain counts
print("=== URL Examples by Subdomain Count ===\n")

for num_subdomains in range(0, 5):
    print(f"\n--- {num_subdomains} Subdomain(s) ---")
    
    # Get samples for this subdomain count
    samples = df4[df4['NoOfSubDomain'] == num_subdomains][['URL', 'Domain', 'NoOfSubDomain', 'label']].head(2)
    
    if len(samples) > 0:
        for _, row in samples.iterrows():
            label_str = "LEGIT" if row['label'] == 1 else "PHISH"
            print(f"[{label_str}] {row['Domain']}")
            print(f"        Full URL: {row['URL'][:60]}...")
    else:
        print(f"No URLs with {num_subdomains} subdomains")

# Check for high subdomain counts
print("\n--- High Subdomain Counts (5+) ---")
high_subdomain = df4[df4['NoOfSubDomain'] >= 5][['URL', 'Domain', 'NoOfSubDomain', 'label']]
print(f"Total URLs with 5+ subdomains: {len(high_subdomain)}")
print(f"  Phishing: {len(high_subdomain[high_subdomain['label']==0])}")
print(f"  Legitimate: {len(high_subdomain[high_subdomain['label']==1])}")

if len(high_subdomain) > 0:
    print("\nExamples:")
    for _, row in high_subdomain.head(3).iterrows():
        label_str = "LEGIT" if row['label'] == 1 else "PHISH"
        print(f"[{label_str}] {row['NoOfSubDomain']} subdomains: {row['Domain']}")

In [ ]:
# Verify exact subdomain statistics for documentation
phishing_subs = df4[df4['label'] == 0]['NoOfSubDomain']
legit_subs = df4[df4['label'] == 1]['NoOfSubDomain']

print("=== Exact NoOfSubDomain Statistics ===")
print(f"\nPhishing URLs (label=0):")
print(f"  Mean: {phishing_subs.mean():.2f}")
print(f"  Median: {phishing_subs.median():.0f}")
print(f"  Max: {phishing_subs.max()}")

print(f"\nLegitimate URLs (label=1):")
print(f"  Mean: {legit_subs.mean():.2f}")
print(f"  Median: {legit_subs.median():.0f}")
print(f"  Max: {legit_subs.max()}")

print(f"\n=== 5+ Subdomains Check ===")
five_plus = df4[df4['NoOfSubDomain'] >= 5]
print(f"Total with 5+ subdomains: {len(five_plus)}")
print(f"  Phishing: {len(five_plus[five_plus['label']==0])}")
print(f"  Legitimate: {len(five_plus[five_plus['label']==1])}")
print(f"  Percentage phishing: {len(five_plus[five_plus['label']==0])/len(five_plus)*100:.1f}%")

#### NoOfSubDomain Findings

**Initial assumption**: I assumed phishing URLs would use more subdomains for deception (like `paypal.com.secure.evil.com`).

**Actual findings from data** (verified in cell above):
- **Phishing**: Mean 1.17, Median 1, Max 10 subdomains
- **Legitimate**: Mean 1.16, Median 1, Max 4 subdomains
- Both distributions are nearly identical on average (both ~1.16-1.17 mean)

**Critical discovery**: **URLs with 5+ subdomains are ALWAYS phishing!**
- 371 URLs have 5+ subdomains
- **100% are phishing** (0 legitimate)
- Examples show classic phishing patterns:
  - `www.amazon.co.jp.infotoop.shop` (mimicking Amazon Japan)
  - `www.dlrect-smtb.jp.ap1.ib.commetryx.com` (fake Japanese bank)

**Subdomain distribution insights**:
- 0 subdomains: Mostly phishing (bare domains like `ipfs.io`)
- 1-2 subdomains: Common for both (standard `www.domain.com`)
- 3-4 subdomains: Mixed usage
- 5+ subdomains: **100% phishing** (perfect indicator!)

**Why phishers use many subdomains**:
- Create deceptive URLs that look legitimate at first glance
- Hide the real domain at the end (`amazon.co.jp.evil.com`)
- Exploit users who only check the beginning of URLs

**Implication for rule-based system**:
- `if NoOfSubDomain >= 5: classify as PHISHING` (100% precision)
- Another perfect indicator to add to our rule set
- Combined with other rules, we're catching more phishing patterns

### Feature Investigation: Domain

I assume there is no correlation between a domain name and whether it's phishing or legitimate - anyone can register any available domain name. Let's test this assumption with the data.

In [ ]:
# Investigate Domain feature
print("=== Domain Feature Overview ===\n")

# First, see what Domain actually contains
print("Sample Domain values:")
print(df4['Domain'].head(10).to_string(index=False))

print(f"\n=== Domain Statistics ===")
print(f"Total URLs: {len(df4):,}")
print(f"Unique domains: {df4['Domain'].nunique():,}")

print(f"\nPhishing URLs (label=0): {len(df4[df4['label']==0]):,}")
print(f"Unique phishing domains: {df4[df4['label']==0]['Domain'].nunique():,}")

print(f"\nLegitimate URLs (label=1): {len(df4[df4['label']==1]):,}")
print(f"Unique legitimate domains: {df4[df4['label']==1]['Domain'].nunique():,}")

# Check for domain overlap between classes
phishing_domains = set(df4[df4['label']==0]['Domain'])
legit_domains = set(df4[df4['label']==1]['Domain'])
overlap = phishing_domains.intersection(legit_domains)

print(f"\n=== Domain Overlap Between Classes ===")
print(f"Domains appearing in BOTH classes: {len(overlap):,}")

if len(overlap) > 0:
    print(f"\nExample domains in both classes:")
    for domain in list(overlap)[:10]:
        phish_count = len(df4[(df4['Domain']==domain) & (df4['label']==0)])
        legit_count = len(df4[(df4['Domain']==domain) & (df4['label']==1)])
        print(f"  {domain}: {phish_count} phishing, {legit_count} legitimate")

In [ ]:
# Verify exact Domain statistics for documentation
print("=== Exact Domain Statistics for Verification ===\n")

phishing_df = df4[df4['label'] == 0]
legit_df = df4[df4['label'] == 1]

print(f"Phishing (label=0):")
print(f"  Total URLs: {len(phishing_df):,}")
print(f"  Unique domains: {phishing_df['Domain'].nunique():,}")
print(f"  Ratio: {len(phishing_df) / phishing_df['Domain'].nunique():.2f} URLs per domain")

print(f"\nLegitimate (label=1):")
print(f"  Total URLs: {len(legit_df):,}")
print(f"  Unique domains: {legit_df['Domain'].nunique():,}")
print(f"  Ratio: {len(legit_df) / legit_df['Domain'].nunique():.2f} URLs per domain")

# Overlap details
phishing_domains = set(phishing_df['Domain'])
legit_domains = set(legit_df['Domain'])
overlap = phishing_domains.intersection(legit_domains)

print(f"\nDomain Overlap:")
print(f"  Domains in BOTH classes: {len(overlap)}")
print(f"  Percentage of unique domains: {len(overlap) / df4['Domain'].nunique() * 100:.2f}%")

# Top phishing domain
top_phishing = phishing_df['Domain'].value_counts().iloc[0]
top_phishing_domain = phishing_df['Domain'].value_counts().index[0]
print(f"\nMost reused phishing domain:")
print(f"  {top_phishing_domain}: {top_phishing:,} URLs")

# Check legitimate domain uniqueness
legit_max_count = legit_df['Domain'].value_counts().max()
print(f"\nLegitimate domain reuse:")
print(f"  Maximum URLs per domain: {legit_max_count}")

In [ ]:
# Look deeper at domain reuse patterns
print("=== Domain Reuse Patterns ===\n")

# Top domains by frequency (phishing)
print("Top 10 phishing domains by frequency:")
phishing_domain_counts = df4[df4['label']==0]['Domain'].value_counts().head(10)
for domain, count in phishing_domain_counts.items():
    print(f"  {domain}: {count} URLs")

print("\nTop 10 legitimate domains by frequency:")
legit_domain_counts = df4[df4['label']==1]['Domain'].value_counts().head(10)
for domain, count in legit_domain_counts.items():
    print(f"  {domain}: {count} URLs")

# Examine domains that appear in both classes more closely
print("\n=== Domains in BOTH Classes (all 54) ===")
overlap_analysis = []
for domain in overlap:
    phish_count = len(df4[(df4['Domain']==domain) & (df4['label']==0)])
    legit_count = len(df4[(df4['Domain']==domain) & (df4['label']==1)])
    total = phish_count + legit_count
    overlap_analysis.append({
        'Domain': domain,
        'Phishing': phish_count,
        'Legitimate': legit_count,
        'Total': total
    })

overlap_df = pd.DataFrame(overlap_analysis).sort_values('Total', ascending=False)
display(overlap_df.head(20).style
        .set_caption("Domains Appearing in Both Classes")
        .format({'Phishing': '{:,}', 'Legitimate': '{:,}', 'Total': '{:,}'}))

#### Domain Findings

**Initial assumption**: I assumed there is no correlation between domain names and phishing/legitimate classification.

**Actual findings from data** (verified in cell above):
- **Phishing**: 100,945 URLs from 85,290 unique domains (1.18 URLs per domain)
- **Legitimate**: 134,850 URLs from 134,850 unique domains (1.00 URLs per domain - 100% unique!)
- **Domain overlap**: Only 54 domains (0.02%) appear in both classes

**Critical discovery**: Legitimate sites have COMPLETELY unique domains (every URL = different domain), while phishing sites REUSE certain domains extensively!

**Top reused phishing domains**:
- `ipfs.io`: 1,197 phishing URLs (file storage)
- `docs.google.com`: 526 phishing URLs (Google Docs)
- `cloudflare-ipfs.com`, `gateway.ipfs.io`, `s3.amazonaws.com`: Cloud storage platforms

**Why this pattern exists**:
- **Legitimate sites**: Dataset contains diverse legitimate websites, each with its own unique domain
- **Phishing attacks**: Heavily exploit cloud storage and file-sharing platforms to host phishing pages (free, trusted domains, hard to block entirely)
- Phishers abuse legitimate platforms (Google Docs, IPFS, AWS S3) to gain trust from the platform's reputation

**Implication for classification**:
- Domain NAME alone isn't useful for classification (anyone can register any domain)
- However, **certain domains are phishing hotspots** (ipfs.io, docs.google.com, etc.)
- The 54 domains appearing in both classes confirm Domain can't be a reliable sole indicator
- Domain uniqueness pattern is a dataset artifact, not a generalizable feature

### Feature Investigation: CharContinuationRate

First, I need to understand what CharContinuationRate actually measures. Let me examine its definition and test its usefulness for our challenge.

**What CharContinuationRate actually measures:**

CharContinuationRate is a mathematical metric based on **bigram probabilities** learned from legitimate URLs. The formula is:

$$\text{CharContinuationRate} = \exp\left(\frac{1}{n-1} \sum_{i=1}^{n-1} \log P(c_{i+1} \mid c_i)\right)$$

Where:
- $c_i$ = the i-th character in the URL (usually lowercased)
- $n$ = length of the URL string
- $P(c_{i+1} \mid c_i)$ = probability of character $c_{i+1}$ appearing after $c_i$, based on a bigram model trained on millions of legitimate URLs

**IMPORTANT**: This is calculated on the **entire URL** (scheme + subdomain + domain + path + query + fragment).

**How values are determined:**
- **1.0**: All character pairs (bigrams) in the URL are common/normal in legitimate URLs
- **0.0**: At least ONE character pair was never seen in the training corpus (log(0) = -∞ → entire score becomes 0)
- **0.5-0.9**: Mix of common and uncommon character transitions

**Examples:**
- `southbankmosaics.com` = 1.0: All letter transitions are common English patterns
- `www.dzwww.com` = 0.0: Contains a character bigram never seen in training data (unusual repetition pattern)
- `f0519141.xsph.ru` = Low score: Random mixing of letters/digits creates unusual character transitions

In [ ]:
# Test CharContinuationRate for discriminative power
print("=== CharContinuationRate Statistics ===\n")

# Overall statistics (does it help in general?)
phishing_char = df4[df4['label'] == 0]['CharContinuationRate']
legit_char = df4[df4['label'] == 1]['CharContinuationRate']

print("Overall Statistics:")
print(f"Phishing: Mean {phishing_char.mean():.4f}, Median {phishing_char.median():.4f}")
print(f"Legitimate: Mean {legit_char.mean():.4f}, Median {legit_char.median():.4f}")
print(f"Difference: {legit_char.mean() - phishing_char.mean():.4f}")

# Test on shared domains (our specific challenge)
phishing_domains = set(df4[df4['label']==0]['Domain'])
legit_domains = set(df4[df4['label']==1]['Domain'])
overlap = list(phishing_domains.intersection(legit_domains))

print(f"\n=== Test on Shared Domains (Our Challenge) ===")
print(f"Testing {len(overlap)} domains appearing in BOTH classes:\n")

for domain in overlap[:10]:
    phish_rate = df4[(df4['Domain']==domain) & (df4['label']==0)]['CharContinuationRate'].mean()
    legit_rate = df4[(df4['Domain']==domain) & (df4['label']==1)]['CharContinuationRate'].mean()
    diff = abs(phish_rate - legit_rate)
    print(f"{domain}: Phish {phish_rate:.4f}, Legit {legit_rate:.4f}, Diff {diff:.4f}")

#### CharContinuationRate Findings

**When this feature DOES work** (verified in cell above):
- **General phishing detection**: Legitimate URLs have significantly higher mean (0.9332 vs 0.7284 for phishing)
- This is because legitimate sites use recognizable domain names with common English letter sequences (`southbankmosaics.com`, `uni-mainz.de`)
- Phishing sites often use random/generated domains with unusual character transitions (`f0519141.xsph.ru`, `kuradox92.lima-city.de`)
- The feature HAS discriminative power for overall classification

**When this feature FAILS** (our specific challenge):
- **Phishing on legitimate platforms**: The 54 domains appearing in both classes show minimal or no difference in CharContinuationRate
- Our key challenge: ipfs.io (1,197 phishing URLs), docs.google.com (526 phishing URLs)

**Why it fails - the mathematical reason**:
CharContinuationRate is calculated on the **entire URL string**. When phishing exploits a legitimate domain:
- Example: `https://docs.google.com/presentation/d/[malicious-content]`
- The legitimate portion (`https://docs.google.com`) contains extremely high-probability English bigrams
- These high-probability bigrams **drown out any weirdness** in the path or query parameters
- Result: Phishing URLs on legitimate platforms still score 0.94-0.99 — **indistinguishable from legitimate content**

**Conclusion**: 
- CharContinuationRate is **useful for general ML model training** (contributes to overall detection)
- CharContinuationRate is **not useful for our specific business challenge** (detecting phishing on legitimate platforms)
- The legitimate domain portion dominates the calculation, masking suspicious patterns in paths/parameters

**Implication**: We need features that analyze **path, parameters, or page content separately** from the domain to solve our shared-domain challenge.